# Module 6 — Capstone: Multi-Polygon Anomaly Candidate Detector

**Goal:** combine every prior module into one small, real, end-to-end pipeline: given a handful of field polygons and a year of imagery, produce a table of candidate vegetation-anomaly events per polygon — a scaled-down, runnable version of the glyphosate spray-detection candidate generator.

**Time:** ~2-3 hours. This is meant to feel like real project work, not another set of guided exercises — you'll be assembling pieces from Modules 1-5 with much less hand-holding.

### What you're building
1. A small `FeatureCollection` of field-like polygons (stand-ins for COMTRS chunks — swap in your real asset later)
2. A cleaned, cloud-masked Sentinel-2 collection with NDVI computed
3. A per-polygon harmonic seasonal baseline fit across multiple years
4. Per-polygon, per-date residuals extracted via `reduceRegions` (not a point loop)
5. A simple candidate-flagging rule (recall-first, per your project's stated priority): flag any date where the residual z-score drops below some threshold
6. Output: one long-format DataFrame + one summary table of flagged candidate events per polygon, ready to hand to a downstream classifier


In [ ]:
import ee
try:
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')

import pandas as pd
import numpy as np


### Step 1 — Polygons

Swap this for `ee.FeatureCollection('projects/YOUR-PROJECT/assets/comtrs_california')` (with `id_field='CO_MTRS'`) when you move this to the real project.

In [ ]:
def make_box(lon, lat, size_deg=0.014):
    return ee.Geometry.Rectangle([lon - size_deg, lat - size_deg, lon + size_deg, lat + size_deg])

chunks = ee.FeatureCollection([
    ee.Feature(make_box(-121.5, 38.5), {'CO_MTRS': 'CHUNK_A'}),
    ee.Feature(make_box(-121.3, 38.6), {'CO_MTRS': 'CHUNK_B'}),
    ee.Feature(make_box(-121.7, 38.4), {'CO_MTRS': 'CHUNK_C'}),
])
id_field = 'CO_MTRS'


### Step 2 — Cleaned NDVI collection with harmonic time bands

TODO — pull this together from Modules 2, 3 and 5: cloud mask, NDVI, `t`/`cos`/`sin`/`constant` bands, filtered to a multi-year window and `chunks.geometry().bounds()`.

In [ ]:
def mask_s2_clouds(image):
    # TODO (Module 2)
    pass

def add_ndvi(image):
    # TODO (Module 1/2)
    pass

def add_harmonic_bands(image, epoch='2019-01-01'):
    # TODO (Module 5)
    pass

# s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
#       .filterDate(...)
#       .filterBounds(chunks)
#       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))
#       .map(mask_s2_clouds)
#       .map(add_ndvi)
#       .map(add_harmonic_bands))


### Step 3 — Fit the harmonic regression, build predicted + residual bands

TODO — from Module 5: `ee.Reducer.linearRegression`, then a `.map()` that adds `predicted_NDVI` and `residual` bands to every image.

In [ ]:
indep_vars = ['constant', 't', 'cos', 'sin']

# harmonic_regression = s2.select(indep_vars + ['NDVI']).reduce(
#     ee.Reducer.linearRegression(numX=len(indep_vars), numY=1)
# )
# coeffs = harmonic_regression.select('coefficients')

def predict_and_residual(image):
    # TODO
    pass

# s2_scored = s2.map(predict_and_residual)


### Step 4 — Extract residuals per polygon per date, at scale

TODO — from Module 4: `map` over images, `reduceRegions` over `chunks` inside each, `.flatten()` into one long `FeatureCollection`/DataFrame with columns `date`, `id_field`, `residual` (and ideally pixel `count` so you can drop low-coverage observations).

In [ ]:
def reduce_one_image(image):
    # TODO
    pass

# long_fc = s2_scored.map(reduce_one_image).flatten()
# records = long_fc.getInfo()['features']
# long_df = pd.DataFrame([r['properties'] for r in records]).dropna(subset=['residual'])
# long_df['date'] = pd.to_datetime(long_df['date'])


### Step 5 — Recall-first candidate flagging

Per your project's stated priority (catch every true event, accept extra false positives), a permissive rule is appropriate here: flag any polygon-date where the residual is more than *k* standard deviations below that polygon's own residual distribution, with a fairly generous `k` (e.g. 1.5–2, not 3+). Compute per-polygon residual mean/std with pandas `.groupby(id_field)`, then flag.

In [ ]:
def flag_candidates(long_df, id_field, k=1.5):
    stats = long_df.groupby(id_field)['residual'].agg(['mean', 'std']).rename(
        columns={'mean': 'resid_mean', 'std': 'resid_std'})
    merged = long_df.merge(stats, on=id_field)
    merged['zscore'] = (merged['residual'] - merged['resid_mean']) / merged['resid_std']
    merged['candidate'] = merged['zscore'] < -k
    return merged

# scored_df = flag_candidates(long_df, id_field, k=1.5)
# summary = scored_df.groupby(id_field)['candidate'].sum().rename('n_candidate_dates')
# summary


### Step 6 — Wire it all together

Now fill in the TODOs above end-to-end and run the full pipeline on the toy `chunks`. Plot the residual/z-score time series for each chunk with candidate dates marked, and sanity-check: do the flagged dates look like plausible short-duration dips rather than just noisy single points from a cloud-mask miss? If you're getting too many false positives from single noisy observations, consider requiring the dip to persist across ≥2 consecutive clear observations before flagging — a cheap precision improvement that doesn't sacrifice much recall.


In [ ]:
# TODO: run the full pipeline and plot results


### Where this goes next (beyond this notebook)

This capstone deliberately stops at "candidate events per polygon" — exactly where your project notes say the current step ends, before candidates get fed into the trained random forest. From here, the natural next steps (outside the scope of a GEE-specific curriculum, more general ML engineering) are:

- Swap the toy `chunks` for your real ~39MB COMTRS asset, uploaded once as a GEE Asset (`earthengine upload table`), and switch synchronous `.getInfo()` calls to batched `Export.table.toDrive`/`toCloudStorage` runs, tiled spatially and/or temporally to stay under request limits (Module 5, Exercise 5.1)
- Feed the candidate-event windows (with surrounding pixel-level context, not just the polygon mean) into your MIL or RF model as instances
- Consider whether the harmonic baseline should be refit per crop-type or refit whenever a crop rotation is detected (a large, sustained shift in baseline NDVI level is a very different signal from a short glyphosate-like dip) — worth a follow-up experiment once you're back in your own repo

---
**You're done with the structured curriculum.** At this point you have the actual GEE vocabulary — `Image`/`ImageCollection`/`FeatureCollection`, `.map()`, `reduceRegion`/`reduceRegions`, harmonic regression via `linearRegression`, and async `Export` — to read and extend your own glyphosate pipeline code without it feeling like a black box.